In [1]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

processed_dir = '../data/processed/'
df = pd.read_csv(os.path.join(processed_dir, 'featured_orders.csv'))

print("Data loaded successfully! Shape:", df.shape)
display(df.head())

Data loaded successfully! Shape: (20639, 15)


,orderid,customerid,restaurantid,deliverypartnerid,orderdate,ordertime,deliverytimeminutes,foodcost,deliveryfee,discount,couponcode,gst,finalamount,orderstatus,paymentmethod
0,-1.731787,0.323840,-0.370233,-1.528447,0.348597,-0.550921,0.185955,-0.983226,0.810685,1.024723,0.332852,-1.123440,-1.462329,-0.127907,-1.461025
1,-1.731618,1.013188,0.646418,1.337020,0.362810,-0.851310,-0.723592,0.285461,-0.478151,1.404867,-0.980881,0.270690,-0.326927,2.141005,-0.285289
2,-1.731449,1.455541,-1.429962,-1.691198,-0.060559,0.944566,0.440628,-0.513985,0.166267,-0.128121,0.778379,-0.607803,-0.547946,-1.262363,0.302579
3,-1.731280,-0.979122,0.919248,-1.112911,1.344580,-2.718246,-0.323391,-1.829018,-0.155942,-0.573432,0.778379,-2.052860,-1.862123,-0.127907,-0.873157
4,-1.731111,0.667797,0.968070,-1.460922,-0.353973,-0.434641,-0.468919,1.194977,1.390661,1.684156,-0.398269,1.270135,0.749701,-0.127907,0.890447


In [2]:
if 'delivery_time' in df.columns:
    X = df.drop(columns=['delivery_time'])
    y = df['delivery_time']
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print("Data split into training and testing sets successfully.")
else:
    print("Target column 'delivery_time' not found in dataset.")

Target column 'delivery_time' not found in dataset.


In [3]:
target_col = 'deliverytimeminutes'

if target_col in df.columns:
    X = df.drop(columns=[target_col])
    y = df[target_col]
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print("Data split into training and testing sets successfully.")
else:
    print(f"Target column '{target_col}' not found in dataset.")

Data split into training and testing sets successfully.


In [4]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    
    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2 Score": r2
    })

results_df = pd.DataFrame(results)
print("--- Delivery Model Performance Comparison ---")
print(results_df)

--- Delivery Model Performance Comparison ---
               Model       MAE      RMSE  R2 Score
0  Linear Regression  0.475064  1.015858  0.018314
1      Random Forest  0.437953  0.976258  0.093358


In [5]:
os.makedirs('../models', exist_ok=True)

best_model = models["Random Forest"]
joblib.dump(best_model, '../models/delivery_model.pkl')
print("Best delivery model successfully saved to '../models/delivery_model.pkl'!")

Best delivery model successfully saved to '../models/delivery_model.pkl'!
